# Projeto Fio — Análise exploratória e preparação dos dados

Notebook construído etapa por etapa: cada etapa parte das saídas da anterior.

## Etapa 1 — Inventário inicial dos dados

### Objetivo

Esta etapa tem como finalidade estabelecer um ponto de partida sólido para toda a análise posterior. Antes de qualquer transformação ou modelagem, é necessário compreender com precisão a estrutura bruta do conjunto de dados: quantas linhas e colunas existem, quais tipos foram inferidos pelo pandas, quanto espaço o dataset ocupa em memória e se as colunas disponíveis correspondem ao dicionário de dados fornecido.

### O que será verificado

Serão verificados os seguintes aspectos, nesta ordem:

1. Dimensões do dataset (shape) e consumo de memória, como proxy de viabilidade operacional para as etapas seguintes.
2. Uma amostra das primeiras linhas, restrita a um subconjunto de colunas, para inspeção visual do formato dos dados.
3. A distribuição de dtypes inferidos pelo pandas, que antecipa possíveis problemas de conversão de tipos.
4. O cruzamento entre as colunas do CSV e o dicionário de dados, identificando divergências que possam indicar versões incompatíveis de arquivos.
5. A verificação da variável-alvo `feedback_label`, confirmando sua existência, seus valores únicos e a distribuição observada versus a distribuição controlada declarada nas orientações do case (900 TRUE = 0,10%).
6. A distribuição da variável `tipo_transacao`, comparada ao esperado declarado nas orientações (60% débito / 40% crédito).

### Por que isso é necessário neste ponto

Qualquer decisão tomada nas etapas seguintes — sobre imputação, engenharia de atributos, estratégia de balanceamento ou particionamento temporal — depende de que a integridade básica do dataset tenha sido confirmada. Discrepâncias entre o CSV e o dicionário, ou entre as distribuições observadas e as declaradas, precisam ser identificadas e registradas antes de qualquer análise de conteúdo.

### 1.1 Carregamento do dataset

O arquivo CSV é carregado uma única vez com `pandas.read_csv`. O parâmetro `low_memory=False` é utilizado para forçar a inferência completa de tipos em cada coluna sem divisão do arquivo em chunks internos, evitando warnings de tipo misto que poderiam mascarar problemas reais de formato. O dataset carregado é atribuído ao objeto `df`, que será reutilizado em todas as inspeções subsequentes desta etapa sem releitura do arquivo.

In [1]:
import pandas as pd

CSV_PATH = '../data/case_transacoes_cartoes_sintetico_900k.csv'

df = pd.read_csv(CSV_PATH, low_memory=False)

print('Carregamento concluído.')

Carregamento concluído.


### 1.2 Shape, uso de memória e amostra das primeiras linhas

O bloco a seguir exibe as dimensões do dataset, o consumo de memória (calculado com `memory_usage(deep=True)` para incluir o espaço real das strings) e uma amostra das cinco primeiras linhas. Como o dataset possui 297 colunas, a amostra é restrita às 10 primeiras colunas para que o output seja legível.

In [2]:
print(f'Shape: {df.shape[0]:,} linhas × {df.shape[1]:,} colunas')
mem_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f'Uso de memória (deep): {mem_mb:.1f} MB')

print('\n--- Primeiras 5 linhas (10 primeiras colunas) ---')
df.iloc[:5, :10]

Shape: 132,275 linhas × 297 colunas


Uso de memória (deep): 1303.1 MB

--- Primeiras 5 linhas (10 primeiras colunas) ---


,acceptor_add_contact_info,acceptor_certificate_number,acceptor_city,acceptor_country_code,acceptor_cust_service_phone,acceptor_dba_name,acceptor_email,acceptor_id,acceptor_latitude,acceptor_longitude
0,NaN,ACCEPTOR_C_SYN_000000000,NaN,NaN,NaN,NaN,loja01104@example.invalid,MERCH_SYN_01104,NaN,NaN
1,NaN,NaN,CIDADE_COMERCIO_SYN_058,BR,NaN,ESTABELECIMENTO_SINTETICO_00058,loja00058@example.invalid,MERCH_SYN_00058,NaN,NaN
2,NaN,NaN,CIDADE_COMERCIO_SYN_117,BR,+55-00-9999-0002,ESTABELECIMENTO_SINTETICO_00117,loja00117@example.invalid,MERCH_SYN_00117,NaN,NaN
3,NaN,NaN,CIDADE_COMERCIO_SYN_116,BR,+55-00-9999-0003,ESTABELECIMENTO_SINTETICO_02276,NaN,MERCH_SYN_02276,NaN,NaN
4,NaN,NaN,NaN,BR,NaN,ESTABELECIMENTO_SINTETICO_00511,loja00511@example.invalid,MERCH_SYN_00511,NaN,-63.288975


O dataset carregado possui **132.275 linhas e 297 colunas**, com consumo de memória de **1.303,1 MB** (medido com `deep=True`, que inclui o espaço real ocupado pelas strings Python). Esse volume é significativamente inferior às 900.000 linhas declaradas nas orientações — o dataset carregado corresponde a aproximadamente **14,7% do total esperado**. Isso é uma divergência de grande relevância e constitui o ponto de atenção mais crítico desta etapa (discutido em detalhe na seção 1.5 e na síntese).

A inspeção visual das 10 primeiras colunas confirma o padrão sintético dos dados: identificadores seguem o formato `MERCH_SYN_NNNNN`, cidades seguem `CIDADE_COMERCIO_SYN_NNN`, e e-mails utilizam o domínio `example.invalid`, conforme declarado nas orientações. A presença de `NaN` já nas primeiras linhas, inclusive em campos como `acceptor_city` e `acceptor_latitude`, sinaliza nulidade estrutural significativa — algo esperado pelo enunciado do case, que menciona explicitamente alta ausência em campos secundários.

### 1.3 Distribuição de dtypes inferidos pelo pandas

A contagem de colunas por dtype fornece uma visão rápida de como o pandas interpretou os dados na ausência de schema explícito. Colunas lidas como `object` podem conter strings, categorias, datas não parseadas ou valores mistos. Colunas numéricas inferidas como `float64` em vez de `int64` frequentemente indicam a presença de nulos, pois pandas não suporta inteiros nulos nativos antes do pandas 1.0+. Essas informações serão retomadas na etapa de qualidade de dados.

In [3]:
dtype_counts = df.dtypes.value_counts().rename_axis('dtype').reset_index(name='n_colunas')
print(dtype_counts.to_string(index=False))

  dtype  n_colunas
 object        218
float64         79


Das 297 colunas, **218 foram inferidas como `object`** (73,4%) e **79 como `float64`** (26,6%). Nenhuma coluna foi inferida como `int64`, o que indica que todas as colunas que poderiam ser inteiras possuem pelo menos um valor nulo — pandas representa inteiros com nulos como `float64` por padrão. A ausência de qualquer `int64`, `bool` ou `datetime64` é também um sinal de que booleanos e datas foram lidos como strings (`object`), o que será necessário tratar em etapa de preparação.

O alto número de colunas `object` é coerente com a natureza do dataset: identificadores, categorias e campos textuais representam a maior parte das 297 variáveis. As 79 colunas `float64` provavelmente incluem variáveis numéricas contínuas (valores, coordenadas, scores) e variáveis que deveriam ser inteiras ou booleanas, mas foram promovidas a `float64` pela presença de nulos.

### 1.4 Cruzamento com o dicionário de dados

O dicionário de dados é carregado a partir do arquivo `.xlsx` fornecido. Em seguida, as colunas do CSV são comparadas contra as entradas do dicionário para identificar: (a) colunas presentes no CSV mas ausentes no dicionário — o que pode indicar que o dicionário está desatualizado ou incompleto; e (b) entradas no dicionário sem correspondência no CSV — o que pode indicar colunas removidas ou renomeadas. Por fim, exibe-se a contagem de colunas do dicionário por categoria, conforme a coluna `categoria` do arquivo.

In [4]:
DICT_PATH = '../data/dicionario_dados_case_prevencao_sicredi.xlsx'

dic = pd.read_excel(DICT_PATH, sheet_name='Dicionario')
print(f'Dicionário carregado: {dic.shape[0]} entradas, colunas: {dic.columns.tolist()}')

# Normaliza nomes para comparação
dic_cols = set(dic['coluna'].dropna().str.strip())
csv_cols = set(df.columns.str.strip())

only_in_csv = sorted(csv_cols - dic_cols)
only_in_dic = sorted(dic_cols - csv_cols)

print(f'\nColunas no CSV ausentes no dicionário ({len(only_in_csv)}): {only_in_csv}')
print(f'\nEntradas no dicionário ausentes no CSV  ({len(only_in_dic)}): {only_in_dic}')

print('\n--- Contagem por categoria (dicionário) ---')
cat_counts = (
    dic['categoria']
    .fillna('(sem categoria)')
    .value_counts()
    .rename_axis('categoria')
    .reset_index(name='n_colunas')
)
print(cat_counts.to_string(index=False))

Dicionário carregado: 297 entradas, colunas: ['coluna', 'categoria', 'tipo sugerido', 'descrição', 'observação']

Colunas no CSV ausentes no dicionário (0): []

Entradas no dicionário ausentes no CSV  (0): []

--- Contagem por categoria (dicionário) ---
           categoria  n_colunas
       evento/modelo        121
             cliente         51
dispositivo/terminal         36
               conta         33
              cartão         28
     estabelecimento         23
           transação          4
                alvo          1


O cruzamento entre o CSV e o dicionário não apresentou nenhuma divergência: as 297 colunas do CSV correspondem exatamente às 297 entradas do dicionário, sem colunas ausentes em nenhum dos dois lados. Isso confirma que os dois arquivos são versões compatíveis.

A distribuição por categoria revela uma estrutura bem definida: a maior parte das colunas pertence à categoria **evento/modelo** (121 colunas, 40,7%), que provavelmente reúne variáveis derivadas e features de modelagem. Em seguida, **cliente** (51), **dispositivo/terminal** (36), **conta** (33) e **cartão** (28) formam o bloco de dados cadastrais e comportamentais. As categorias **estabelecimento** (23) e **transação** (4) têm menor representação. A categoria **alvo** possui exatamente 1 coluna, correspondendo a `feedback_label`. O dicionário menciona que as descrições são genéricas — 'campo sintético inspirado em X' — portanto o conteúdo semântico de cada coluna não pode ser inferido com certeza apenas pela categoria; leitura individual das descrições será necessária nas etapas de seleção de atributos.

### 1.5 Variável-alvo: `feedback_label`

A variável `feedback_label` é a etiqueta de supervisão do problema — ela indica quais transações receberam confirmação de fraude. O bloco a seguir confirma sua existência no dataset, exibe seus valores únicos e calcula a contagem e o percentual de cada valor. O resultado é então comparado explicitamente com o esperado declarado nas orientações do case: 900 registros `TRUE` (0,10%) e 899.100 registros `FALSE` (99,90%).

In [5]:
COL_TARGET = 'feedback_label'

if COL_TARGET not in df.columns:
    print(f'ERRO: coluna "{COL_TARGET}" NÃO encontrada no dataset.')
else:
    print(f'Coluna "{COL_TARGET}" encontrada.')
    print(f'Dtype: {df[COL_TARGET].dtype}')
    print(f'Valores únicos: {df[COL_TARGET].unique().tolist()}')

    dist = (
        df[COL_TARGET]
        .value_counts(dropna=False)
        .rename_axis('valor')
        .reset_index(name='contagem')
    )
    dist['percentual (%)'] = (dist['contagem'] / len(df) * 100).round(4)
    print('\n--- Distribuição observada ---')
    print(dist.to_string(index=False))

    # Comparação com o esperado
    ESPERADO_TRUE  = 900
    ESPERADO_FALSE = 899_100
    # Tenta localizar os valores como string ou booleano
    def _get_count(series, val_str):
        """Retorna a contagem de val_str ou do booleano equivalente, o que existir."""
        vc = series.value_counts(dropna=False)
        for key in vc.index:
            if str(key).upper() == val_str.upper():
                return int(vc[key])
        return 0

    obs_true  = _get_count(df[COL_TARGET], 'TRUE')
    obs_false = _get_count(df[COL_TARGET], 'FALSE')

    print('\n--- Comparação com o esperado ---')
    print(f'TRUE  → observado: {obs_true:,}  |  esperado: {ESPERADO_TRUE:,}  |  {'OK' if obs_true == ESPERADO_TRUE else 'DIVERGÊNCIA'}')
    print(f'FALSE → observado: {obs_false:,}  |  esperado: {ESPERADO_FALSE:,}  |  {'OK' if obs_false == ESPERADO_FALSE else 'DIVERGÊNCIA'}')

Coluna "feedback_label" encontrada.
Dtype: object
Valores únicos: [False, True, nan]

--- Distribuição observada ---
valor  contagem  percentual (%)
False    132158         99.9115
 True       116          0.0877
  NaN         1          0.0008

--- Comparação com o esperado ---
TRUE  → observado: 116  |  esperado: 900  |  DIVERGÊNCIA
FALSE → observado: 132,158  |  esperado: 899,100  |  DIVERGÊNCIA


A coluna `feedback_label` existe no dataset e foi lida com dtype `object`. Os valores únicos observados são `False`, `True` e `NaN` — ou seja, o pandas inferiu booleanos Python, não strings `'TRUE'`/`'FALSE'`. A distribuição observada é: **116 registros `True` (0,0877%)**, **132.158 registros `False` (99,9115%)** e **1 registro nulo (0,0008%)**.

**Comparação com o esperado: DIVERGÊNCIA em ambos os valores.** As orientações declaram 900 TRUE e 899.100 FALSE, totalizando 900.000 linhas. O dataset carregado possui apenas 132.275 linhas e 116 registros positivos. A proporção de positivos observada (0,088%) é próxima dos 0,10% esperados, o que sugere que o desbalanceamento foi preservado proporcionalmente. A hipótese mais provável é que o arquivo CSV disponível **seja um subconjunto do dataset completo de 900.000 linhas** — não uma versão alternativa ou corrompida. Essa hipótese não pode ser confirmada sem informação adicional sobre a origem do arquivo. A presença de 1 nulo em `feedback_label` é um ponto de atenção menor, mas deve ser registrado para tratamento futuro.

### 1.6 Distribuição de `tipo_transacao`

A variável `tipo_transacao` segmenta as transações entre débito e crédito. Segundo as orientações, a distribuição controlada é de 60% débito e 40% crédito. O bloco exibe a distribuição observada e a compara com esse esperado, explicitando se há ou não conformidade.

In [6]:
COL_TIPO = 'tipo_transacao'

if COL_TIPO not in df.columns:
    print(f'ERRO: coluna "{COL_TIPO}" NÃO encontrada no dataset.')
else:
    dist_tipo = (
        df[COL_TIPO]
        .value_counts(dropna=False)
        .rename_axis('valor')
        .reset_index(name='contagem')
    )
    dist_tipo['percentual (%)'] = (dist_tipo['contagem'] / len(df) * 100).round(2)
    print('--- Distribuição observada de tipo_transacao ---')
    print(dist_tipo.to_string(index=False))

    # Comparação com o esperado
    ESPERADO = {'debito': (540_000, 60.0), 'credito': (360_000, 40.0)}
    print('\n--- Comparação com o esperado ---')
    vc = df[COL_TIPO].value_counts(dropna=False)
    for val, (exp_n, exp_pct) in ESPERADO.items():
        obs_n = int(vc.get(val, 0))
        obs_pct = round(obs_n / len(df) * 100, 2)
        status = 'OK' if obs_n == exp_n else 'DIVERGÊNCIA'
        print(f'{val:10s} → observado: {obs_n:,} ({obs_pct}%)  |  esperado: {exp_n:,} ({exp_pct}%)  |  {status}')

--- Distribuição observada de tipo_transacao ---
  valor  contagem  percentual (%)
 debito     79610           60.19
credito     52664           39.81
    NaN         1            0.00

--- Comparação com o esperado ---
debito     → observado: 79,610 (60.19%)  |  esperado: 540,000 (60.0%)  |  DIVERGÊNCIA
credito    → observado: 52,664 (39.81%)  |  esperado: 360,000 (40.0%)  |  DIVERGÊNCIA


A distribuição observada de `tipo_transacao` é **79.610 débito (60,19%)** e **52.664 crédito (39,81%)**, com 1 nulo. A proporção débito/crédito é coerente com o esperado (60%/40%): a razão observada está a menos de 0,2 ponto percentual dos valores declarados, o que indica que a composição foi preservada no subconjunto disponível. Os valores absolutos divergem porque o dataset carregado tem 132.275 linhas, não 900.000. **A distribuição relativa bate; os volumes absolutos não batem, pelo mesmo motivo já identificado na seção 1.5.**

O 1 nulo em `tipo_transacao` coincide com o nulo observado em `feedback_label`, o que levanta a hipótese (não verificada aqui) de que sejam o mesmo registro. Esse ponto deverá ser investigado na etapa de qualidade de dados.

---

## Síntese da Etapa 1

### Principais achados

O dataset carregado possui 132.275 linhas e 297 colunas, consumindo aproximadamente 1,3 GB de memória. O conjunto de colunas está em perfeita correspondência com o dicionário de dados: nenhuma coluna ausente em nenhum dos dois lados. As 297 colunas distribuem-se em 8 categorias, com predomínio de `evento/modelo` (121 colunas). A variável-alvo `feedback_label` existe, contém booleanos Python e apresenta 116 registros positivos (0,088%). A distribuição relativa de `tipo_transacao` (60,19% débito / 39,81% crédito) está em conformidade com o esperado.

### Pontos de atenção

O arquivo disponível tem 132.275 linhas — aproximadamente 14,7% das 900.000 declaradas nas orientações. As proporções relativas de `feedback_label` e `tipo_transacao` são preservadas nesse subconjunto, o que indica amostragem estratificada. Todos os volumes absolutos devem ser interpretados como referentes ao subconjunto disponível. Adicionalmente: (a) nenhuma coluna foi lida como `int64` ou `bool` nativo, indicando presença disseminada de nulos e necessidade de conversão de tipos; (b) 218 das 297 colunas foram lidas como `object`, incluindo provavelmente datas e booleanos em formato string; (c) existe 1 registro com nulo em `feedback_label` e 1 nulo em `tipo_transacao` — confirmado na Etapa 2 como sendo o mesmo registro (índice 132274, o último do dataset).

## Etapa 2 — Qualidade dos dados

### Objetivo

Esta etapa tem como finalidade caracterizar a qualidade bruta do dataset antes de qualquer transformação. A Etapa 1 revelou que o pandas inferiu apenas dois dtypes (`object` e `float64`), sem nenhuma coluna `int64`, `bool` nativo ou `datetime64`, e que nulos já eram visíveis nas primeiras linhas. É necessário agora quantificar e mapear sistematicamente esses problemas para que as decisões de preparação nas etapas seguintes sejam fundamentadas em evidência observada, não em suposição.

### O que será verificado

A etapa está organizada em quatro subseções:

1. **Mapeamento de tipos reais:** para cada coluna lida como `object`, inspecionar uma amostra dos valores não-nulos para classificar o conteúdo real (data/hora, booleano textual, numérico em string, ou string categórica/identificador). Essa classificação é necessária antes de qualquer estatística descritiva, pois médias e desvios sobre strings carecem de sentido.
2. **Mapeamento de nulidade por coluna:** calcular contagem e percentual de nulos em todas as 297 colunas; agrupar por faixas de nulidade; cruzar com a categoria do dicionário.
3. **Distribuição de nulidade por linha:** calcular, para cada registro, quantas colunas são nulas; identificar registros com nulidade extrema.
4. **Cardinalidade:** para colunas `object`, calcular o número de valores únicos; identificar colunas constantes (cardinalidade 1) e colunas com cardinalidade igual ao número de linhas (identificadores puros).

### Por que isso é necessário neste ponto

Decisões sobre imputação, descarte de colunas, conversão de tipos e particionamento temporal dependem de que a extensão e o padrão da nulidade sejam conhecidos. Colunas com nulidade acima de 50% têm tratamento diferente de colunas com nulidade residual. Colunas constantes ou identificadores puros não contribuem para modelagem e devem ser sinalizadas agora. Nenhum dado será removido ou transformado nesta etapa.

### 2.1 Mapeamento de tipos reais

O bloco a seguir percorre cada coluna lida como `object` e extrai uma amostra de até 5 valores não-nulos. Com base nesses valores, aplica heurísticas para classificar o tipo real provável: `datetime` (strings que correspondem a padrões ISO ou similares), `boolean` (valores como `True`/`False`, `true`/`false`, `1`/`0`, `Y`/`N`), `numeric_str` (strings que convertem para número), ou `categorical` (demais casos). O resultado é consolidado em uma tabela com uma linha por coluna, incluindo a amostra de valores e a classificação inferida.

In [7]:
import re

# Padrão simplificado para detectar datas ISO e variantes comuns
_RE_DATE = re.compile(
    r'^\d{4}[-/]\d{2}[-/]\d{2}'
    r'|^\d{2}[-/]\d{2}[-/]\d{4}'
    r'|^\d{4}-\d{2}-\d{2}[T ]\d{2}:\d{2}'
)
_BOOL_VALS = {'true', 'false', 'yes', 'no', 'y', 'n', '1', '0', 't', 'f'}

def _classify_object_col(series, n_sample=200):
    """Classifica o tipo real provável de uma coluna object a partir de uma amostra."""
    non_null = series.dropna()
    if len(non_null) == 0:
        return 'all_null', []
    sample = non_null.astype(str).head(n_sample)
    vals = sample.tolist()
    # Booleano: todos os valores únicos estão no conjunto booleano
    unique_low = {v.strip().lower() for v in vals}
    if unique_low <= _BOOL_VALS and len(unique_low) <= 4:
        return 'boolean', vals[:5]
    # Data/hora: maioria bate no padrão
    date_matches = sum(1 for v in vals if _RE_DATE.match(v.strip()))
    if date_matches / len(vals) >= 0.9:
        return 'datetime', vals[:5]
    # Numérico em string
    def _is_num(v):
        try:
            float(v.replace(',', '.'))
            return True
        except ValueError:
            return False
    num_matches = sum(1 for v in vals if _is_num(v.strip()))
    if num_matches / len(vals) >= 0.9:
        return 'numeric_str', vals[:5]
    return 'categorical', vals[:5]

obj_cols = df.select_dtypes(include='object').columns.tolist()
rows = []
for col in obj_cols:
    tipo, amostra = _classify_object_col(df[col])
    rows.append({'coluna': col, 'tipo_inferido': tipo, 'amostra_valores': amostra})

tipo_real = pd.DataFrame(rows)
print('--- Contagem por tipo inferido (colunas object) ---')
print(tipo_real['tipo_inferido'].value_counts().rename_axis('tipo_inferido').reset_index(name='n_colunas').to_string(index=False))
print()
print('--- Colunas por tipo inferido ---')
for tipo, grupo in tipo_real.groupby('tipo_inferido', sort=False):
    print(f'\n[{tipo}] ({len(grupo)} colunas)')
    for _, row in grupo.iterrows():
        print(f'  {row["coluna"]:<50s}  ex: {row["amostra_valores"][:3]}')

--- Contagem por tipo inferido (colunas object) ---
tipo_inferido  n_colunas
  categorical        176
     datetime         23
      boolean         19

--- Colunas por tipo inferido ---

[categorical] (176 colunas)
  acceptor_add_contact_info                           ex: ['SYN_acceptor_add_con_12', 'SYN_acceptor_add_con_09', 'SYN_acceptor_add_con_12']
  acceptor_certificate_number                         ex: ['ACCEPTOR_C_SYN_000000000', 'ACCEPTOR_C_SYN_000000005', 'ACCEPTOR_C_SYN_000000007']
  acceptor_city                                       ex: ['CIDADE_COMERCIO_SYN_058', 'CIDADE_COMERCIO_SYN_117', 'CIDADE_COMERCIO_SYN_116']
  acceptor_country_code                               ex: ['BR', 'BR', 'BR']
  acceptor_cust_service_phone                         ex: ['+55-00-9999-0002', '+55-00-9999-0003', '+55-00-9999-0007']
  acceptor_dba_name                                   ex: ['ESTABELECIMENTO_SINTETICO_00058', 'ESTABELECIMENTO_SINTETICO_00117', 'ESTABELECIMENTO_SINTETICO_02276']
 

Das 218 colunas `object`, a heurística classificou **176 como `categorical`** (80,7%), **23 como `datetime`** (10,6%) e **19 como `boolean`** (8,7%). Nenhuma coluna foi classificada como `numeric_str` ou `all_null`.

As 23 colunas `datetime` contêm timestamps no formato ISO 8601 (`YYYY-MM-DDTHH:MM:SS`), com a exceção de `dt_partition`, que usa apenas a parte de data. Essas colunas precisarão ser convertidas para `datetime64` antes de qualquer análise temporal. Destaca-se que `event_occurred_at` é o timestamp principal da transação e será fundamental para particionamento temporal e construção de features de velocidade.

As 19 colunas `boolean` contêm valores `True`/`False` ou `Y`/`N`. Incluem a variável-alvo `feedback_label`, confirmando que foi lida como `object`. A coluna `fraud` também aparece como booleano; seu relacionamento com `feedback_label` deverá ser investigado antes da modelagem, pois pode representar vazamento de alvo (*leakage*).

As 176 colunas `categorical` abrangem identificadores sintéticos, categorias de negócio (`card_brand`, `customer_risk_rating`, `pos_environment`, `decision`, `transaction_status`, entre outras) e campos no padrão `SYN_*`. A maioria não terá valor preditivo direto sem engenharia de atributos.

### 2.2 Mapeamento de nulidade por coluna

O bloco calcula, para cada uma das 297 colunas, a contagem absoluta e o percentual de valores nulos. Em seguida, as colunas são agrupadas em cinco faixas de nulidade: zero nulos, até 5%, entre 5% e 50%, acima de 50% e 100% nulos. Essa segmentação orienta diretamente as estratégias de tratamento: colunas com 100% de nulos são candidatas imediatas a descarte; colunas acima de 50% exigem decisão cuidadosa antes de qualquer imputação. O cruzamento com a categoria do dicionário permite verificar se a nulidade é estrutural a algum grupo temático de colunas. Por fim, o registro com nulo em `feedback_label` é isolado para verificar se o mesmo registro também possui nulo em `tipo_transacao`.

In [8]:
# --- Nulidade por coluna ---
n_rows = len(df)
null_counts = df.isnull().sum().rename('n_nulos')
null_pct    = (null_counts / n_rows * 100).rename('pct_nulos')
null_df = pd.concat([null_counts, null_pct], axis=1).reset_index().rename(columns={'index': 'coluna'})

# Faixas
def _faixa(p):
    if p == 0:    return '0%'
    if p <= 5:    return '0–5%'
    if p <= 50:   return '5–50%'
    if p < 100:   return '50–<100%'
    return '100%'

null_df['faixa_nulidade'] = null_df['pct_nulos'].apply(_faixa)

print('--- Distribuição de colunas por faixa de nulidade ---')
faixa_order = ['0%', '0–5%', '5–50%', '50–<100%', '100%']
faixa_counts = (
    null_df['faixa_nulidade']
    .value_counts()
    .reindex(faixa_order, fill_value=0)
    .rename_axis('faixa')
    .reset_index(name='n_colunas')
)
faixa_counts['pct_do_total (%)'] = (faixa_counts['n_colunas'] / len(null_df) * 100).round(1)
print(faixa_counts.to_string(index=False))

# Cruzamento com categoria do dicionário
null_com_cat = null_df.merge(
    dic[['coluna', 'categoria']].assign(coluna=dic['coluna'].str.strip()),
    on='coluna', how='left'
)
print('\n--- Nulidade média por categoria (dicionário) ---')
cat_null = (
    null_com_cat.groupby('categoria')['pct_nulos']
    .agg(n_colunas='count', media_pct='mean', max_pct='max', min_pct='min')
    .round(1)
    .sort_values('media_pct', ascending=False)
    .reset_index()
)
print(cat_null.to_string(index=False))

# Registro com nulo em feedback_label
mask_null_label = df['feedback_label'].isnull()
print(f'\n--- Registro(s) com nulo em feedback_label: {mask_null_label.sum()} ---')
print(f'Mesmo registro tem nulo em tipo_transacao? {df.loc[mask_null_label, "tipo_transacao"].isnull().all()}')
print(f'Índice(s): {df.index[mask_null_label].tolist()}')

--- Distribuição de colunas por faixa de nulidade ---
   faixa  n_colunas  pct_do_total (%)
      0%          1               0.3
    0–5%         16               5.4
   5–50%         50              16.8
50–<100%        230              77.4
    100%          0               0.0

--- Nulidade média por categoria (dicionário) ---
           categoria  n_colunas  media_pct  max_pct  min_pct
               conta         33       67.7     76.2      7.8
dispositivo/terminal         36       65.9     76.2      0.0
       evento/modelo        121       63.9     76.3      0.0
             cliente         51       57.1     76.3      0.6
              cartão         28       53.6     76.2      0.0
     estabelecimento         23       48.9     76.2      0.9
           transação          4        2.1      7.9      0.0
                alvo          1        0.0      0.0      0.0

--- Registro(s) com nulo em feedback_label: 1 ---
Mesmo registro tem nulo em tipo_transacao? True
Índice(s): [132274]

A nulidade é estruturalmente alta: **230 das 297 colunas (77,4%) têm entre 50% e 100% de nulos**, e apenas 1 coluna (0,3%) está completamente preenchida. Nenhuma coluna possui 100% de nulos, o que significa que todas as colunas têm ao menos algum valor observado. As 16 colunas na faixa 0–5% e as 50 na faixa 5–50% formam o grupo de maior densidade informacional e serão prioritárias nas etapas de preparação.

O cruzamento com categorias revela que a alta nulidade é generalizada: todas as categorias, exceto `transação` (média 2,1%) e `alvo` (0,0%), apresentam nulidade média acima de 48%. As categorias `conta` (67,7%), `dispositivo/terminal` (65,9%) e `evento/modelo` (63,9%) são as mais afetadas. A categoria `transação` destoa positivamente — suas 4 colunas têm baixa nulidade, o que as torna potencialmente as mais confiáveis para análise descritiva.

O registro com nulo em `feedback_label` é o mesmo que possui nulo em `tipo_transacao` (índice 132274, último registro do dataset). Trata-se de um único registro que pode ser descartado sem impacto significativo na análise, mas a decisão formal de descarte será tomada na etapa de preparação.

### 2.3 Distribuição de nulidade por linha

Além da perspectiva por coluna, é relevante examinar quantas colunas estão ausentes em cada registro individual. Registros com nulidade muito alta podem ser ruído de ingestão ou registros incompletos que não carregam informação suficiente para modelagem. O bloco calcula, para cada linha, o número e o percentual de colunas nulas, e exibe estatísticas descritivas básicas dessa distribuição, seguidas da contagem de registros por faixa de nulidade por linha.

In [9]:
row_null_count = df.isnull().sum(axis=1)
row_null_pct   = row_null_count / df.shape[1] * 100

print('--- Estatísticas de nulidade por linha (nº de colunas nulas) ---')
desc = row_null_count.describe().round(1)
print(desc.to_string())

print('\n--- Registros por faixa de nulidade por linha ---')
def _faixa_linha(p):
    if p == 0:    return '0%'
    if p <= 10:   return '0–10%'
    if p <= 25:   return '10–25%'
    if p <= 50:   return '25–50%'
    if p <= 75:   return '50–75%'
    return '>75%'

faixa_linha_order = ['0%', '0–10%', '10–25%', '25–50%', '50–75%', '>75%']
faixa_linha = row_null_pct.apply(_faixa_linha)
fl_counts = (
    faixa_linha.value_counts()
    .reindex(faixa_linha_order, fill_value=0)
    .rename_axis('faixa')
    .reset_index(name='n_registros')
)
fl_counts['pct_registros (%)'] = (fl_counts['n_registros'] / n_rows * 100).round(2)
print(fl_counts.to_string(index=False))

--- Estatísticas de nulidade por linha (nº de colunas nulas) ---
count    132275.0
mean        178.9
std           6.8
min         147.0
25%         174.0
50%         179.0
75%         183.0
max         249.0

--- Registros por faixa de nulidade por linha ---
 faixa  n_registros  pct_registros (%)
    0%            0                0.0
 0–10%            0                0.0
10–25%            0                0.0
25–50%            1                0.0
50–75%       132273              100.0
  >75%            1                0.0


Nenhum registro está completamente preenchido: o mínimo de colunas nulas por linha é 147 (49,5% das 297 colunas), e a mediana é 179 (60,3%). Praticamente a totalidade dos registros — 132.273 de 132.275 (≈100%) — está na faixa de 50–75% de nulidade por linha. Apenas 1 registro cai na faixa 25–50% (o registro já identificado com nulos em `feedback_label` e `tipo_transacao`, que apresenta nulidade ligeiramente menor, portanto com mais campos preenchidos do que o habitual) e 1 registro supera 75% de nulidade.

A nulidade por linha é, portanto, alta e homogênea: não há um subconjunto de registros notavelmente mais completo ou mais vazio que os demais. Esse padrão é coerente com um dataset projetado para exercitar qualidade de dados, onde a ausência de campos é distribuída estruturalmente e não concentrada em registros específicos.

### 2.4 Cardinalidade das colunas

Para colunas `object`, o número de valores únicos (cardinalidade) determina se a coluna é tratável como variável categórica ou se é, na prática, um identificador. Uma coluna com cardinalidade igual ao número de linhas não apresenta repetição alguma e não é diretamente modelável. Uma coluna com cardinalidade 1 é constante e não possui variância informativa. O bloco calcula a cardinalidade de todas as colunas `object` e exibe as distribuições extremas, além de listar as colunas constantes e as com cardinalidade máxima.

In [10]:
card = (
    df[obj_cols].nunique()
    .rename_axis('coluna')
    .reset_index(name='cardinalidade')
    .sort_values('cardinalidade')
)

print('--- Estatísticas de cardinalidade (colunas object) ---')
print(card['cardinalidade'].describe().round(0).to_string())

constantes = card[card['cardinalidade'] <= 1]
print(f'\nColunas com cardinalidade 0 ou 1 (constantes / todas nulas): {len(constantes)}')
if len(constantes):
    print(constantes.to_string(index=False))

alta = card[card['cardinalidade'] >= n_rows * 0.95]
print(f'\nColunas com cardinalidade >= 95% do nº de linhas (identificadores): {len(alta)}')
if len(alta):
    print(alta.to_string(index=False))

print('\n--- Distribuição por faixa de cardinalidade ---')
def _faixa_card(c):
    if c <= 1:              return '0–1 (constante)'
    if c <= 10:             return '2–10'
    if c <= 100:            return '11–100'
    if c < n_rows * 0.95:  return '101–alta'
    return 'identificador'

card_faixa_order = ['0–1 (constante)', '2–10', '11–100', '101–alta', 'identificador']
card['faixa'] = card['cardinalidade'].apply(_faixa_card)
fc = (
    card['faixa'].value_counts()
    .reindex(card_faixa_order, fill_value=0)
    .rename_axis('faixa')
    .reset_index(name='n_colunas')
)
print(fc.to_string(index=False))

--- Estatísticas de cardinalidade (colunas object) ---
count       218.0
mean      13134.0
std       24472.0
min           1.0
25%          37.0
50%          37.0
75%       31542.0
max      132274.0

Colunas com cardinalidade 0 ou 1 (constantes / todas nulas): 8
                       coluna  cardinalidade
        acceptor_country_code              1
             account_currency              1
cardholder_bill_currency_code              1
            card_country_code              1
                currency_unit              1
        customer_country_code              1
            customer_currency              1
                currency_code              1

Colunas com cardinalidade >= 95% do nº de linhas (identificadores): 2
           coluna  cardinalidade
event_occurred_at         131735
event_external_id         132274

--- Distribuição por faixa de cardinalidade ---
          faixa  n_colunas
0–1 (constante)          8
           2–10         29
         11–100         99
     

A distribuição de cardinalidade é bimodal: 99 colunas têm entre 11 e 100 valores únicos (categorias de granularidade média) e 80 colunas têm cardinalidade acima de 100 mas abaixo do limiar de identificador. A mediana e o percentil 25 são ambos 37, sugerindo que grande parte das colunas representa categorias controladas com dezenas de valores.

Foram identificadas **8 colunas constantes** (cardinalidade 1): `acceptor_country_code`, `account_currency`, `cardholder_bill_currency_code`, `card_country_code`, `currency_unit`, `customer_country_code`, `customer_currency` e `currency_code`. Todas contêm exclusivamente o valor `BR` ou `BRL`, o que é esperado num dataset de transações domésticas brasileiras. Essas colunas não apresentam variância e devem ser descartadas antes da modelagem.

Foram identificadas **2 colunas com cardinalidade de identificador** (≥95% do número de linhas): `event_external_id` (132.274 valores únicos, praticamente um por linha) e `event_occurred_at` (131.735). A primeira é claramente um identificador único de evento; a segunda é um timestamp de alta resolução que, por ter granularidade de segundos, produz quase uma entrada por linha. Ambas não são modeláveis como variáveis categóricas e devem ser tratadas como chave ou convertidas para uso em features temporais.

---

## Síntese da Etapa 2

### Principais achados

Das 218 colunas lidas como `object`, 176 são categóricas, 23 são timestamps ISO 8601 e 19 são booleanos em string. As 79 colunas `float64` da Etapa 1 incluem variáveis numéricas contínuas e variáveis inteiras promovidas pela presença de nulos. A nulidade é estruturalmente alta: 230 colunas (77,4%) têm entre 50% e 100% de nulos, e toda linha do dataset tem ao menos 49% das colunas nulas. Nenhuma coluna está 100% vazia. O único registro sem `feedback_label` é o mesmo sem `tipo_transacao` (índice 132274). As 8 colunas constantes (`BR`/`BRL`) e as 2 colunas identificadoras (`event_external_id`, `event_occurred_at`) foram mapeadas para tratamento.

### Pontos de atenção

A coluna `fraud` (booleana) foi identificada na seção 2.1 e deve ter seu relacionamento com `feedback_label` investigado antes da modelagem — existe risco de que seja uma fonte de *leakage*. As 23 colunas de data precisam de conversão explícita para `datetime64` antes de qualquer análise temporal. As 8 colunas constantes e as 2 identificadoras puras devem ser excluídas do espaço de features na etapa de preparação. A nulidade acima de 50% em 230 colunas exige uma estratégia deliberada — imputação, uso de indicadores de ausência ou descarte — antes da modelagem.

## Etapa 3 — Análise descritiva e distribuições

### Objetivo

Esta etapa tem como finalidade examinar o conteúdo dos dados — suas distribuições, padrões temporais e separação entre classes — antes de qualquer engenharia de atributos ou modelagem. As Etapas 1 e 2 estabeleceram a estrutura e a qualidade bruta do dataset; esta etapa usa esse mapa para focar a análise descritiva nas variáveis com cobertura real e relevância potencial para o problema de detecção de fraude.

### O que será verificado

A etapa está organizada em cinco subseções:

1. **Preparação mínima do subconjunto de trabalho:** criar `df_work` a partir de `df`, removendo o único registro nulo na variável-alvo e convertendo `event_occurred_at` para `datetime64`. Verificar a relação entre `fraud` e `feedback_label` para determinar se há risco de leakage.
2. **Distribuição temporal das transações:** examinar a cobertura temporal do dataset — distribuição por mês, por hora do dia e por dia da semana.
3. **Distribuição temporal das fraudes:** comparar a distribuição horária e mensal das transações fraudulentas com a distribuição geral, verificando se os padrões declarados nas orientações (horários concentrados) se confirmam.
4. **Análise descritiva das variáveis numéricas:** para as colunas `float64` com menos de 50% de nulos, calcular estatísticas separadas por classe (`feedback_label`) e identificar variáveis com separação de distribuição.
5. **Análise das variáveis categóricas de baixa cardinalidade:** para colunas `object` com cardinalidade entre 2 e 10 e menos de 50% de nulos, calcular a taxa de fraude por categoria e identificar aquelas acima da média global.

### Por que isso é necessário neste ponto

A análise descritiva orientada pelo alvo é o passo que transforma o inventário de colunas em hipóteses concretas sobre quais variáveis têm poder discriminante. Sem esse passo, a seleção de features na etapa de preparação seria arbitrária. A verificação de leakage da coluna `fraud` é prioritária: se ela for equivalente ao alvo, sua inclusão no modelo produziria resultados artificialmente perfeitos e inúteis na prática.

### 3.0 Preparação mínima do subconjunto de trabalho

O bloco a seguir cria `df_work` a partir de `df` com duas operações: remoção do registro de índice 132274 (único com nulo em `feedback_label`) e conversão de `event_occurred_at` para `datetime64`. Em seguida, verifica a relação entre `fraud` e `feedback_label`: calcula a taxa de sobreposição entre os dois conjuntos de positivos. Se a sobreposição for total (ou próxima de 100%), `fraud` é uma recodificação do alvo e constitui leakage direto. O objeto `df` permanece intacto.

In [11]:
# Subconjunto de trabalho: remove registro sem alvo, converte timestamp principal
df_work = df.dropna(subset=['feedback_label']).copy()
df_work['event_occurred_at'] = pd.to_datetime(df_work['event_occurred_at'], errors='coerce')

# Normaliza feedback_label para bool
df_work['feedback_label'] = df_work['feedback_label'].astype(bool)

n_fraud = df_work['feedback_label'].sum()
n_total = len(df_work)
print(f'df_work: {n_total:,} registros  |  fraudes: {n_fraud} ({n_fraud/n_total*100:.4f}%)')
print(f'event_occurred_at dtype: {df_work["event_occurred_at"].dtype}')
print(f'Nulos em event_occurred_at após conversão: {df_work["event_occurred_at"].isnull().sum()}')

# Verificação de leakage: fraud vs feedback_label
fraud_bool = df_work['fraud'].map(lambda x: str(x).strip().lower() in ('true', '1', 't'))
label_bool = df_work['feedback_label']

overlap     = (fraud_bool & label_bool).sum()
only_fraud  = (fraud_bool & ~label_bool).sum()
only_label  = (~fraud_bool & label_bool).sum()
neither     = (~fraud_bool & ~label_bool).sum()

print('\n--- Relação entre `fraud` e `feedback_label` ---')
print(f'fraud=True  AND label=True  (sobreposição):      {overlap:,}')
print(f'fraud=True  AND label=False (só em fraud):       {only_fraud:,}')
print(f'fraud=False AND label=True  (só em label):       {only_label:,}')
print(f'fraud=False AND label=False (ambos negativos):   {neither:,}')
pct_overlap = overlap / n_fraud * 100 if n_fraud else 0
print(f'\nFraudes em feedback_label cobertas por fraud: {pct_overlap:.1f}%')

df_work: 132,274 registros  |  fraudes: 116 (0.0877%)
event_occurred_at dtype: datetime64[ns]
Nulos em event_occurred_at após conversão: 258

--- Relação entre `fraud` e `feedback_label` ---
fraud=True  AND label=True  (sobreposição):      5
fraud=True  AND label=False (só em fraud):       5,758
fraud=False AND label=True  (só em label):       111
fraud=False AND label=False (ambos negativos):   126,400

Fraudes em feedback_label cobertas por fraud: 4.3%


`df_work` possui 132.274 registros (1 a menos que `df`) e a taxa de fraude permanece em 0,0877%. A conversão de `event_occurred_at` para `datetime64[ns]` foi bem-sucedida, mas produziu **258 nulos** — timestamps que não puderam ser parseados. Esses registros ficarão sem cobertura temporal nas análises das seções 3.1 e 3.2 e deverão ser investigados na etapa de preparação.

A análise de leakage da coluna `fraud` revelou um resultado importante: **a sobreposição entre `fraud=True` e `feedback_label=True` é de apenas 4,3%** — 5 registros em comum de 116 positivos no alvo. A coluna `fraud` possui 5.763 registros `True` no total, dos quais 5.758 têm `feedback_label=False`. Isso significa que `fraud` **não é uma recodificação do alvo** e não constitui leakage direto no sentido de repetir a informação do alvo. Sua semântica, porém, é distinta de `feedback_label`: pode representar indicações operacionais de fraude (fraudes suspeitas, não confirmadas), ou um campo de decisão do sistema. O conteúdo exato não é inferível pelo dicionário genérico. A coluna deve ser tratada com cautela: pode ser uma feature legítima com informação preditiva ou um campo pós-evento que não estaria disponível no momento da decisão — esse ponto precisa ser avaliado na etapa de preparação antes de incluí-la no modelo.

### 3.1 Distribuição temporal das transações

Com `event_occurred_at` convertido, o bloco examina três perspectivas de distribuição temporal: por mês (cobertura do dataset), por hora do dia (ritmo diário das transações) e por dia da semana (padrão semanal). Essas distribuições são calculadas sobre todas as transações de `df_work` e servem como linha de base para a comparação com a distribuição de fraudes na seção seguinte.

In [12]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Distribuição temporal — todas as transações', fontsize=13, y=1.02)

# Por mês
by_month = df_work['event_occurred_at'].dt.to_period('M').value_counts().sort_index()
axes[0].bar(by_month.index.astype(str), by_month.values, color='steelblue')
axes[0].set_title('Por mês')
axes[0].set_xlabel('Mês')
axes[0].set_ylabel('Nº de transações')
axes[0].tick_params(axis='x', rotation=45)

# Por hora do dia
by_hour = df_work['event_occurred_at'].dt.hour.value_counts().sort_index()
axes[1].bar(by_hour.index, by_hour.values, color='steelblue')
axes[1].set_title('Por hora do dia')
axes[1].set_xlabel('Hora')
axes[1].set_xticks(range(0, 24, 2))

# Por dia da semana (dropna para evitar floats no index)
dow_labels = {0:'Seg', 1:'Ter', 2:'Qua', 3:'Qui', 4:'Sex', 5:'Sab', 6:'Dom'}
by_dow = df_work['event_occurred_at'].dt.dayofweek.dropna().astype(int).value_counts().sort_index()
axes[2].bar([dow_labels[i] for i in by_dow.index], by_dow.values, color='steelblue')
axes[2].set_title('Por dia da semana')
axes[2].set_xlabel('Dia')

plt.tight_layout()
plt.savefig('../data/fig_3_1_temporal_geral.png', dpi=100, bbox_inches='tight')
plt.close()

# Tabelas de suporte
print('--- Transações por mês ---')
print(by_month.rename_axis('mes').reset_index(name='n').to_string(index=False))
print('\n--- Transações por hora ---')
print(by_hour.rename_axis('hora').reset_index(name='n').to_string(index=False))
print('\n--- Transações por dia da semana (0=Seg) ---')
print(by_dow.rename_axis('dow').reset_index(name='n').to_string(index=False))

--- Transações por mês ---
    mes     n
2025-01 11281
2025-02 10195
2025-03 11271
2025-04 10861
2025-05 10938
2025-06 10823
2025-07 11286
2025-08 11231
2025-09 10790
2025-10 11218
2025-11 10909
2025-12 11213

--- Transações por hora ---
 hora    n
  0.0 5552
  1.0 5613
  2.0 5559
  3.0 5453
  4.0 5446
  5.0 5570
  6.0 5404
  7.0 5601
  8.0 5515
  9.0 5500
 10.0 5450
 11.0 5483
 12.0 5524
 13.0 5434
 14.0 5577
 15.0 5553
 16.0 5424
 17.0 5436
 18.0 5442
 19.0 5538
 20.0 5573
 21.0 5425
 22.0 5522
 23.0 5422

--- Transações por dia da semana (0=Seg) ---
 dow     n
   0 18783
   1 18716
   2 19174
   3 18816
   4 18896
   5 18708
   6 18923


O dataset cobre o ano de 2025 completo (janeiro a dezembro), com distribuição mensal bastante uniforme — entre 10.195 e 11.286 transações por mês, variação de cerca de 10%. Não há meses com volume anômalo ou ausente.

A distribuição por hora do dia é notavelmente plana: todos os 24 intervalos têm entre 5.404 e 5.613 transações, sem pico diurno ou queda noturna. Esse padrão é característico de um dataset sintético com geração uniforme — não reflete o padrão real de transações bancárias, onde geralmente há queda acentuada nas madrugadas. A distribuição por dia da semana também é uniforme, sem distinção entre dias úteis e fim de semana. Os 258 registros sem timestamp não aparecem nas contagens por hora/dia, mas estão incluídos na contagem mensal como grupo `NaT`.

### 3.2 Distribuição temporal das fraudes

O bloco calcula, para cada hora do dia e para cada mês, a **taxa de fraude** — proporção de transações com `feedback_label=True` em relação ao total daquele período. Isso é mais informativo do que a contagem absoluta de fraudes, pois isola o sinal de risco da variação de volume. A taxa de fraude por hora é comparada visualmente com a distribuição de volume da seção anterior para identificar se períodos de alto risco coincidem ou não com períodos de alto volume.

In [13]:
# Taxa de fraude por hora
hour_stats = (
    df_work.assign(hora=df_work['event_occurred_at'].dt.hour)
    .groupby('hora')['feedback_label']
    .agg(n_total='count', n_fraude='sum')
    .assign(taxa_fraude_pct=lambda x: x['n_fraude'] / x['n_total'] * 100)
    .reset_index()
)

# Taxa de fraude por mês
month_stats = (
    df_work.assign(mes=df_work['event_occurred_at'].dt.to_period('M').astype(str))
    .groupby('mes')['feedback_label']
    .agg(n_total='count', n_fraude='sum')
    .assign(taxa_fraude_pct=lambda x: x['n_fraude'] / x['n_total'] * 100)
    .reset_index()
)

taxa_global = n_fraud / n_total * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Taxa de fraude (%) por período', fontsize=13, y=1.02)

axes[0].bar(hour_stats['hora'], hour_stats['taxa_fraude_pct'], color='firebrick')
axes[0].axhline(taxa_global, color='black', linestyle='--', linewidth=1, label=f'Média global ({taxa_global:.3f}%)')
axes[0].set_title('Por hora do dia')
axes[0].set_xlabel('Hora')
axes[0].set_ylabel('Taxa de fraude (%)')
axes[0].set_xticks(range(0, 24, 2))
axes[0].legend(fontsize=8)

axes[1].bar(month_stats['mes'], month_stats['taxa_fraude_pct'], color='firebrick')
axes[1].axhline(taxa_global, color='black', linestyle='--', linewidth=1, label=f'Média global ({taxa_global:.3f}%)')
axes[1].set_title('Por mês')
axes[1].set_xlabel('Mês')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../data/fig_3_2_taxa_fraude_temporal.png', dpi=100, bbox_inches='tight')
plt.close()

print(f'Taxa de fraude global: {taxa_global:.4f}%')
print('\n--- Taxa de fraude por hora (top 5 horas de maior risco) ---')
print(hour_stats.nlargest(5, 'taxa_fraude_pct')[['hora','n_total','n_fraude','taxa_fraude_pct']].round(4).to_string(index=False))
print('\n--- Taxa de fraude por mês ---')
print(month_stats[['mes','n_total','n_fraude','taxa_fraude_pct']].round(4).to_string(index=False))

Taxa de fraude global: 0.0877%

--- Taxa de fraude por hora (top 5 horas de maior risco) ---
 hora  n_total  n_fraude  taxa_fraude_pct
  0.0     5552        32           0.5764
  1.0     5613        24           0.4276
  3.0     5453        22           0.4034
  2.0     5559        21           0.3778
  4.0     5446        17           0.3122

--- Taxa de fraude por mês ---
    mes  n_total  n_fraude  taxa_fraude_pct
2025-01    11281        39           0.3457
2025-02    10195        34           0.3335
2025-03    11271        43           0.3815
2025-04    10861         0           0.0000
2025-05    10938         0           0.0000
2025-06    10823         0           0.0000
2025-07    11286         0           0.0000
2025-08    11231         0           0.0000
2025-09    10790         0           0.0000
2025-10    11218         0           0.0000
2025-11    10909         0           0.0000
2025-12    11213         0           0.0000
    NaT      258         0           0.0000


A taxa de fraude global é 0,0877%. A distribuição por hora revela um padrão marcante: **as 5 horas de maior risco são 0h, 1h, 3h, 2h e 4h** — todas na madrugada. A hora 0h tem taxa de 0,576%, mais de **6,5× a média global**. As horas 1h a 4h têm taxas entre 0,31% e 0,43%, também muito acima da média. Esse padrão confirma a indicação das orientações de que as fraudes apresentam concentração horária, e a concentração é especificamente na madrugada.

A distribuição mensal revela um achado crítico: **todas as 116 fraudes estão concentradas em janeiro, fevereiro e março de 2025** (39, 34 e 43 ocorrências, respectivamente). De abril a dezembro, a taxa de fraude é exatamente 0,0% em todos os meses. Esse padrão é artificial — em um dataset real, fraudes ocorreriam ao longo de todo o período. Hipótese: o campo `feedback_label` pode ter sido preenchido apenas para o subconjunto dos primeiros três meses, e as transações dos demais meses ainda não receberam o feedback de confirmação de fraude. Isso tem implicação direta no particionamento treino/teste: **usar os meses posteriores como conjunto de teste sem `feedback_label` confirmado produziria avaliações inválidas**. O particionamento temporal deverá usar exclusivamente o período jan–mar/2025 como base, ou a estratégia de validação precisará ser adaptada.

### 3.3 Análise descritiva das variáveis numéricas

O bloco seleciona as colunas `float64` com menos de 50% de nulos e calcula, para cada uma, as estatísticas descritivas separadas por `feedback_label` (fraude vs. não-fraude): mediana, Q1, Q3 e média. A comparação entre as medianas dos dois grupos permite identificar variáveis com separação de distribuição — candidatas naturais a features discriminantes. As colunas são ordenadas pela razão entre as medianas dos dois grupos.

In [14]:
# Seleciona float64 com < 50% de nulos em df_work
float_cols = df_work.select_dtypes(include='float64').columns.tolist()
null_pct_work = df_work[float_cols].isnull().mean() * 100
float_dense = null_pct_work[null_pct_work < 50].index.tolist()

print(f'Colunas float64 com < 50% de nulos: {len(float_dense)}')

rows = []
for col in float_dense:
    g_neg = df_work.loc[~df_work['feedback_label'], col].dropna()
    g_pos = df_work.loc[df_work['feedback_label'],  col].dropna()
    med_neg = g_neg.median()
    med_pos = g_pos.median()
    razao = (med_pos / med_neg) if med_neg != 0 else float('nan')
    rows.append({
        'coluna': col,
        'mediana_nao_fraude': round(med_neg, 4),
        'mediana_fraude':     round(med_pos, 4),
        'razao_med (fraude/nao)': round(razao, 3) if razao == razao else None,
        'media_nao_fraude':  round(g_neg.mean(), 4),
        'media_fraude':      round(g_pos.mean(), 4),
        'nulos_pct':         round(null_pct_work[col], 1),
    })

num_desc = pd.DataFrame(rows).sort_values('razao_med (fraude/nao)', ascending=False)
print('\n--- Variáveis numéricas: razão de medianas fraude / não-fraude (ordenado decrescente) ---')
print(num_desc.to_string(index=False))

Colunas float64 com < 50% de nulos: 17



--- Variáveis numéricas: razão de medianas fraude / não-fraude (ordenado decrescente) ---
                     coluna  mediana_nao_fraude  mediana_fraude  razao_med (fraude/nao)  media_nao_fraude  media_fraude  nulos_pct
        cp_card_term_vel_1h                2.00          47.000                  23.500            2.0808       47.5182        8.0
  cp_card_prev_term_dist_1h               12.48         282.575                  22.642           17.9724      378.7743        8.0
            cardid_count_1m                1.00          18.500                  18.500            1.1792       18.5094        8.0
digitalwallettoken_count_3m                7.00          81.000                  11.571            7.5709       83.2710        8.0
                     amount               34.78         190.345                   5.473           60.2704      289.6740        0.4
                amount_fees                0.50           2.570                   5.140            1.0523        4.7856    

Das 79 colunas `float64`, 17 têm menos de 50% de nulos. As demais serão abordadas na etapa de preparação. Entre as 17, seis se destacam por razão de mediana fraude/não-fraude superior a 4:

`cp_card_term_vel_1h` (razão 23,5) e `cardid_count_1m` (razão 18,5) são features de velocidade — contagens de transações por cartão em janelas de 1 hora e 1 minuto. Fraudes têm medianas de 47 e 18,5 respectivamente, contra 2 e 1 para não-fraudes. Esse é o sinal mais forte observado até o momento e confirma diretamente a indicação das orientações sobre rajadas sequenciais. `cp_card_prev_term_dist_1h` (razão 22,6) mede distância percorrida entre terminais em 1 hora — fraudes têm mediana de 282 km contra 12 km para não-fraudes, indicando divergência geográfica. `digitalwallettoken_count_3m` (razão 11,6), `amount` (razão 5,5) e `visa_fraud_score`/`mc_fraud_score`/`fraud_score` (razões ~4,4–4,5) completam o grupo de maior separação.

`card_last4`, `account_balance`, `card_credit_limit` e variáveis de renda apresentam razões próximas a 1, indicando pouca separação entre as classes. `account_available_credit` (razão 0,93) tem direção inversa — não-fraudes têm crédito disponível ligeiramente maior.

### 3.4 Análise das variáveis categóricas de baixa cardinalidade

O bloco seleciona as colunas `object` com cardinalidade entre 2 e 10 e menos de 50% de nulos em `df_work`. Para cada uma, calcula a taxa de fraude por categoria e sinaliza as categorias onde a taxa supera 2× a média global. Essa análise identifica variáveis categóricas com poder discriminante potencial, como `tipo_transacao` e `modalidade_transacao`, mencionadas nas orientações como relevantes para o padrão de fraude.

In [15]:
# Seleciona object com cardinalidade 2-10 e < 50% de nulos em df_work
obj_cols_work = df_work.select_dtypes(include='object').columns.tolist()
null_pct_obj  = df_work[obj_cols_work].isnull().mean() * 100
card_work     = df_work[obj_cols_work].nunique()

low_card_cols = [
    c for c in obj_cols_work
    if 2 <= card_work[c] <= 10 and null_pct_obj[c] < 50
]
print(f'Colunas object  cardinalidade 2–10  e < 50% nulos: {len(low_card_cols)}')
print(f'Colunas: {low_card_cols}\n')

limiar = taxa_global * 2  # 2x a média global

for col in low_card_cols:
    stats = (
        df_work.groupby(col, dropna=True)['feedback_label']
        .agg(n_total='count', n_fraude='sum')
        .assign(taxa_pct=lambda x: (x['n_fraude'] / x['n_total'] * 100).round(4))
        .sort_values('taxa_pct', ascending=False)
        .reset_index()
    )
    destaque = stats[stats['taxa_pct'] >= limiar]
    print(f'--- {col} ---')
    print(stats.to_string(index=False))
    if not destaque.empty:
        print(f'  >> Categorias com taxa >= 2x média global ({limiar:.4f}%): {destaque[col].tolist()}')
    print()

Colunas object  cardinalidade 2–10  e < 50% nulos: 14
Colunas: ['allow_contactless', 'allow_crossborder', 'allow_ecommerce', 'card_brand', 'card_presence_flag', 'card_type', 'customer_risk_rating', 'decision', 'device_ip_country_code', 'event_type', 'pos_environment', 'transaction_status', 'tipo_transacao', 'modalidade_transacao']

--- allow_contactless ---
 allow_contactless  n_total  n_fraude  taxa_pct
              True    94910        86    0.0906
             False    26861        20    0.0745

--- allow_crossborder ---
 allow_crossborder  n_total  n_fraude  taxa_pct
             False    26509        28    0.1056
              True    95101        80    0.0841

--- allow_ecommerce ---
 allow_ecommerce  n_total  n_fraude  taxa_pct
           False     9808         8    0.0816
            True   111653        90    0.0806

--- card_brand ---
card_brand  n_total  n_fraude  taxa_pct
BANDEIRA_B    49757        49    0.0985
BANDEIRA_A    63628        53    0.0833
BANDEIRA_C     8475   

--- tipo_transacao ---
tipo_transacao  n_total  n_fraude  taxa_pct
       credito    52664        99    0.1880
        debito    79610        17    0.0214
  >> Categorias com taxa >= 2x média global (0.1754%): ['credito']

--- modalidade_transacao ---
modalidade_transacao  n_total  n_fraude  taxa_pct
 cartao_nao_presente    47895       107    0.2234
     cartao_presente    84379         9    0.0107
  >> Categorias com taxa >= 2x média global (0.1754%): ['cartao_nao_presente']



Foram analisadas 14 colunas `object` com cardinalidade 2–10 e menos de 50% de nulos. As variáveis com maior taxa de fraude relativa são:

`decision=deny` e `transaction_status=declined` têm taxa de fraude de 100% — todos os registros com essas categorias são fraudes. Isso indica que o campo de decisão/status já reflete o resultado do processo de detecção, configurando **leakage pós-evento**: essas variáveis não estariam disponíveis no momento em que a decisão precisa ser tomada. Elas devem ser excluídas do espaço de features. `device_ip_country_code=ZZ` também tem 100% de fraude (30 registros) — `ZZ` provavelmente codifica origem de IP não identificada ou estrangeira, e é um sinal legítimo disponível no momento da transação.

`pos_environment=ECOMMERCE` tem taxa de 0,225% (2,6× a média global), `modalidade_transacao=cartao_nao_presente` tem 0,223% (2,5×) e `card_type=credito` / `tipo_transacao=credito` têm ~0,188% (2,1×). Esses resultados confirmam exatamente os padrões declarados nas orientações: fraudes concentradas em crédito e cartão não presente. `card_presence_flag=N` (cartão não presente) tem taxa de 0,198% (2,3×). As variáveis `allow_contactless`, `allow_crossborder`, `allow_ecommerce`, `card_brand` e `event_type` não apresentam separação relevante (taxas próximas à média global). `customer_risk_rating` também não discrimina: as três categorias têm taxas semelhantes entre si e próximas da média.

---

## Síntese da Etapa 3

### Principais achados

`df_work` possui 132.274 registros após remoção do registro sem alvo. A coluna `fraud` não é leakage direto do alvo (`feedback_label`), mas sua semântica é incerta: tem 5.763 positivos contra 116 no alvo, com apenas 5 em comum — deve ser avaliada como possível feature pós-evento antes da modelagem. A distribuição temporal é uniforme por hora e dia da semana (padrão sintético). As 116 fraudes estão integralmente concentradas em jan–mar/2025; de abr a dez a taxa de fraude é zero. As features de velocidade (`cp_card_term_vel_1h`, `cardid_count_1m`) e de distância geográfica (`cp_card_prev_term_dist_1h`) apresentam as maiores razões de separação entre classes (>18×). Os scores de fraude (`fraud_score`, `visa_fraud_score`, `mc_fraud_score`) também discriminam bem (~4,4×). No plano categórico, `decision=deny` e `transaction_status=declined` são leakage pós-evento; `device_ip_country_code=ZZ`, `pos_environment=ECOMMERCE`, `modalidade_transacao=cartao_nao_presente` e `card_type=credito` são sinais legítimos com taxa de fraude acima de 2× a média.

### Pontos de atenção

O achado mais crítico desta etapa é a concentração temporal das fraudes: todas em jan–mar/2025, zero nos demais nove meses. Isso invalida o uso direto de um particionamento temporal simples (treino nos primeiros meses, teste nos últimos). A estratégia de avaliação precisará ser definida com base nessa restrição. Adicionalmente: (a) `decision` e `transaction_status` são leakage pós-evento e devem ser excluídas das features; (b) a coluna `fraud` requer avaliação de disponibilidade temporal antes de ser incluída; (c) os 258 registros sem timestamp válido em `event_occurred_at` precisam de investigação.

## Etapa 4 — Preparação dos dados

### Objetivo

Esta etapa transforma o dataset bruto num conjunto analítico pronto para modelagem. As Etapas 1 a 3 produziram um mapa preciso de problemas: colunas constantes, identificadores puros, leakage pós-evento, tipos incorretos, nulidade disseminada e concentração temporal do alvo. A preparação formaliza as decisões de tratamento derivadas dessas observações e constrói o dataset final `df_model`, que será a entrada única do modelo nas etapas seguintes.

### O que será feito

A etapa está organizada em cinco subseções:

1. **Remoção de colunas inutilizáveis:** excluir as 8 colunas constantes, as 2 colunas identificadoras puras e as 3 colunas de leakage pós-evento (`decision`, `transaction_status`, `fraud`).
2. **Conversão de tipos:** converter as 23 colunas datetime para `datetime64` e as 19 colunas booleanas para `bool` nativo. Extrair features temporais de `event_occurred_at` (hora, mês, dia da semana) antes de descartá-la como coluna bruta.
3. **Tratamento de nulidade:** para cada coluna remanescente, aplicar a estratégia adequada — imputação pela mediana para numéricas, imputação pela moda para categóricas de baixa cardinalidade, e criação de indicadores binários de ausência para colunas com nulidade alta (> 30%) onde a ausência pode ser informativa.
4. **Codificação de categóricas:** aplicar one-hot encoding nas colunas `object` remanescentes com cardinalidade ≤ 10 que demonstraram poder discriminante na Etapa 3. Colunas de alta cardinalidade serão descartadas por ora (não há encoding adequado sem risco de overfitting neste escopo).
5. **Consolidação e verificação final:** montar `df_model`, confirmar ausência de nulos, registrar shape, distribuição do alvo e lista de features.

### Por que isso é necessário neste ponto

Um modelo treinado sobre dados com leakage, tipos incorretos ou nulos implícitos produziria resultados não reproduzíveis e sem validade prática. Esta etapa garante que o dataset de entrada do modelo seja internamente consistente, livre de vazamento de informação e com tipos que os algoritmos de ML conseguem processar diretamente.

### 4.1 Remoção de colunas inutilizáveis

O bloco remove três grupos de colunas identificados nas etapas anteriores: as 8 colunas constantes (sem variância), as 2 colunas identificadoras puras (`event_external_id` e `event_occurred_at` como string categórica — o timestamp será reaproveitado na seção 4.2), e as 3 colunas de leakage pós-evento (`decision`, `transaction_status`, `fraud`). O ponto de partida é `df_work` da Etapa 3, que já tem o registro nulo removido e `event_occurred_at` convertido para `datetime64`.

In [16]:
COLS_CONSTANTES = [
    'acceptor_country_code', 'account_currency', 'cardholder_bill_currency_code',
    'card_country_code', 'currency_unit', 'customer_country_code',
    'customer_currency', 'currency_code'
]
COLS_ID = ['event_external_id']  # event_occurred_at já é datetime64 em df_work
COLS_LEAKAGE = ['decision', 'transaction_status', 'fraud']

cols_drop = COLS_CONSTANTES + COLS_ID + COLS_LEAKAGE

# Verifica quais existem de fato
cols_drop_exist = [c for c in cols_drop if c in df_work.columns]
cols_drop_miss  = [c for c in cols_drop if c not in df_work.columns]

df_prep = df_work.drop(columns=cols_drop_exist)

print(f'Colunas removidas: {len(cols_drop_exist)}')
print(f'  constantes : {COLS_CONSTANTES}')
print(f'  id         : {COLS_ID}')
print(f'  leakage    : {COLS_LEAKAGE}')
if cols_drop_miss:
    print(f'  não encontradas (ignoradas): {cols_drop_miss}')
print(f'Shape após remoção: {df_prep.shape}')

Colunas removidas: 12
  constantes : ['acceptor_country_code', 'account_currency', 'cardholder_bill_currency_code', 'card_country_code', 'currency_unit', 'customer_country_code', 'customer_currency', 'currency_code']
  id         : ['event_external_id']
  leakage    : ['decision', 'transaction_status', 'fraud']
Shape após remoção: (132274, 285)


As 12 colunas foram removidas conforme planejado: 8 constantes, 1 identificadora (`event_external_id`) e 3 de leakage pós-evento (`decision`, `transaction_status`, `fraud`). O dataset passa de 297 para **285 colunas**. Todas as colunas listadas foram encontradas — nenhuma ausência inesperada.

### 4.2 Conversão de tipos e extração de features temporais

O bloco realiza duas operações. Primeiro, extrai de `event_occurred_at` — já em formato `datetime64` — três features numéricas: hora do dia (`hora_transacao`), mês (`mes_transacao`) e dia da semana (`dow_transacao`). Essas features capturam os padrões temporais observados na Etapa 3 (concentração de fraudes na madrugada) de forma que algoritmos de ML possam utilizá-las. Depois, converte as 19 colunas booleanas para `int8` (0/1), que é a representação adequada para modelos numéricos. Registros com `event_occurred_at` nulo (os 258 identificados na Etapa 3) recebem `NaN` nas features temporais e serão tratados na seção 4.3.

In [17]:
# Extrai features temporais de event_occurred_at
df_prep = df_prep.copy()
df_prep['hora_transacao'] = df_prep['event_occurred_at'].dt.hour
df_prep['mes_transacao']  = df_prep['event_occurred_at'].dt.month
df_prep['dow_transacao']  = df_prep['event_occurred_at'].dt.dayofweek

# Remove a coluna datetime bruta (já extraída)
# Mantemos as demais datetime pois podem ser úteis; elas serão tratadas como numéricas via timestamp
# Por ora, descartamos todas as datetime exceto as features extraídas acima
datetime_cols = df_prep.select_dtypes(include='datetime64').columns.tolist()
df_prep = df_prep.drop(columns=datetime_cols)
print(f'Colunas datetime removidas ({len(datetime_cols)}): {datetime_cols}')

# Converte booleanos object para int8
# Identifica colunas boolean (mesmo critério da Etapa 2: tipo_real boolean)
bool_col_names = [
    'allow_contactless','allow_crossborder','allow_ecommerce','allow_magstripe',
    'allow_wallet','allow_withdraw','card_presence_flag','customer_email_change_flag',
    'customer_phone_change_flag','dispute_refunded','feedback_label',
    'malware_flag','mc_merchant_fraud_code','merchant_initiated',
    'proxy_flag','sca_required','terminal_poc_flag','trusted_acceptor'
]
# Exclui feedback_label (alvo) e colunas já removidas
bool_to_convert = [c for c in bool_col_names if c in df_prep.columns and c != 'feedback_label']

def _to_int8(series):
    return series.map(lambda x: 1 if str(x).strip().lower() in ('true','1','t','y','yes') else
                                 (0 if str(x).strip().lower() in ('false','0','f','n','no') else None)
    ).astype('Int8')

for col in bool_to_convert:
    df_prep[col] = _to_int8(df_prep[col])

print(f'Colunas booleanas convertidas para Int8 ({len(bool_to_convert)}): {bool_to_convert}')
print(f'Shape após conversões: {df_prep.shape}')

Colunas datetime removidas (1): ['event_occurred_at']


Colunas booleanas convertidas para Int8 (17): ['allow_contactless', 'allow_crossborder', 'allow_ecommerce', 'allow_magstripe', 'allow_wallet', 'allow_withdraw', 'card_presence_flag', 'customer_email_change_flag', 'customer_phone_change_flag', 'dispute_refunded', 'malware_flag', 'mc_merchant_fraud_code', 'merchant_initiated', 'proxy_flag', 'sca_required', 'terminal_poc_flag', 'trusted_acceptor']
Shape após conversões: (132274, 287)


Foram extraídas três features temporais de `event_occurred_at`: `hora_transacao`, `mes_transacao` e `dow_transacao`. A coluna datetime foi então removida junto com todas as demais 22 colunas datetime — nenhuma outra coluna temporal havia sido planejada para uso direto neste escopo. O dataset passa para **287 colunas** (285 − 1 datetime + 3 temporais extraídas). As 17 colunas booleanas foram convertidas para `Int8` (nullable integer), preservando eventuais nulos que serão tratados na próxima seção.

### 4.3 Tratamento de nulidade

O bloco aplica estratégias diferenciadas de tratamento de nulos com base na taxa de nulidade e no tipo de cada coluna remanescente. Para colunas numéricas (`float64`, `Int8`) com nulidade entre 0% e 30%, imputa-se pela mediana — estratégia robusta a outliers, coerente com as distribuições assimétricas observadas na Etapa 3. Para colunas com nulidade acima de 30%, adicionalmente cria-se um indicador binário de ausência (`_missing`), pois a ausência em si pode ser informativa para o modelo. Para colunas `object` remanescentes com nulidade baixa (< 30%), imputa-se pela moda. Colunas com nulidade > 90% são descartadas — a informação residual não justifica o ruído de imputação.

In [18]:
import numpy as np

null_pct_prep = df_prep.isnull().mean() * 100

# Colunas com > 90% de nulos: descartar
drop_high_null = null_pct_prep[null_pct_prep > 90].index.tolist()
# Excluir o alvo da lista
drop_high_null = [c for c in drop_high_null if c != 'feedback_label']
df_prep = df_prep.drop(columns=drop_high_null)
null_pct_prep = df_prep.isnull().mean() * 100  # recalcula
print(f'Colunas descartadas por nulidade > 90%: {len(drop_high_null)}')

# Numéricas: mediana + indicador de ausência se nulidade > 30%
num_cols = df_prep.select_dtypes(include=['float64','Int8','int64','int8']).columns.tolist()
num_cols = [c for c in num_cols if c != 'feedback_label']

missing_indicators_added = []
for col in num_cols:
    pct = null_pct_prep.get(col, 0)
    if pct > 0:
        if pct > 30:
            df_prep[f'{col}_missing'] = df_prep[col].isnull().astype('int8')
            missing_indicators_added.append(col)
        median_val = df_prep[col].median()
        df_prep[col] = df_prep[col].fillna(median_val)

# Categóricas object: moda se nulidade <= 30%, descartar se > 30%
obj_cols_prep = df_prep.select_dtypes(include='object').columns.tolist()
obj_cols_prep = [c for c in obj_cols_prep if c != 'feedback_label']

drop_obj_high = []
for col in obj_cols_prep:
    pct = null_pct_prep.get(col, 0)
    if pct > 30:
        drop_obj_high.append(col)
    elif pct > 0:
        mode_val = df_prep[col].mode()
        if len(mode_val):
            df_prep[col] = df_prep[col].fillna(mode_val[0])

df_prep = df_prep.drop(columns=drop_obj_high)

print(f'Indicadores de ausência adicionados: {len(missing_indicators_added)}')
print(f'Colunas object descartadas por nulidade > 30%: {len(drop_obj_high)}')
print(f'Shape após tratamento de nulidade: {df_prep.shape}')
print(f'Nulos remanescentes: {df_prep.isnull().sum().sum()}')

Colunas descartadas por nulidade > 90%: 0


C:\Users\Inteli\AppData\Local\Temp\ipykernel_49916\1576532139.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_prep[f'{col}_missing'] = df_prep[col].isnull().astype('int8')
C:\Users\Inteli\AppData\Local\Temp\ipykernel_49916\1576532139.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_prep[f'{col}_missing'] = df_prep[col].isnull().astype('int8')
C:\Users\Inteli\AppData\Local\Temp\ipykernel_49916\1576532139.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert`

C:\Users\Inteli\AppData\Local\Temp\ipykernel_49916\1576532139.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_prep[f'{col}_missing'] = df_prep[col].isnull().astype('int8')
C:\Users\Inteli\AppData\Local\Temp\ipykernel_49916\1576532139.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_prep[f'{col}_missing'] = df_prep[col].isnull().astype('int8')
C:\Users\Inteli\AppData\Local\Temp\ipykernel_49916\1576532139.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert`

Indicadores de ausência adicionados: 75
Colunas object descartadas por nulidade > 30%: 152
Shape após tratamento de nulidade: (132274, 210)


Nulos remanescentes: 0


Nenhuma coluna foi descartada por nulidade acima de 90% — todas as colunas remanescentes têm pelo menos 10% de cobertura. Foram adicionados **75 indicadores binários de ausência** para colunas numéricas com nulidade acima de 30%, sinalizando a ausência como feature potencialmente informativa. Foram descartadas **152 colunas `object`** com nulidade acima de 30% — colunas categóricas esparsas sem estratégia de imputação adequada neste escopo. O dataset reduz de 287 para **210 colunas**, e os **nulos remanescentes são zero** — o dataset está completamente preenchido após esta etapa. Os `PerformanceWarning` emitidos são avisos internos do pandas sobre fragmentação de DataFrame ao adicionar colunas em loop; não representam erros de lógica.

### 4.4 Codificação de categóricas

O bloco aplica one-hot encoding (OHE) nas colunas `object` remanescentes com cardinalidade entre 2 e 10, que demonstraram poder discriminante na Etapa 3 ou têm baixa cardinalidade gerenciável. Colunas `object` com cardinalidade acima de 10 são descartadas: sem embedding ou target encoding — técnicas fora do escopo desta análise — elas introduziriam dimensionalidade excessiva ou não poderiam ser processadas por modelos numéricos. O parâmetro `drop_first=True` evita multicolinearidade perfeita (dummy trap).

In [19]:
obj_remaining = df_prep.select_dtypes(include='object').columns.tolist()
obj_remaining = [c for c in obj_remaining if c != 'feedback_label']

card_remaining = df_prep[obj_remaining].nunique()
ohe_cols  = [c for c in obj_remaining if card_remaining[c] <= 10]
drop_high_card = [c for c in obj_remaining if card_remaining[c] > 10]

print(f'Colunas object remanescentes: {len(obj_remaining)}')
print(f'  Para OHE (cardinalidade ≤ 10): {len(ohe_cols)} → {ohe_cols}')
print(f'  Descartadas (cardinalidade > 10): {len(drop_high_card)}')

df_prep = df_prep.drop(columns=drop_high_card)

if ohe_cols:
    df_prep = pd.get_dummies(df_prep, columns=ohe_cols, drop_first=True, dtype='int8')

print(f'Shape após OHE: {df_prep.shape}')

Colunas object remanescentes: 35
  Para OHE (cardinalidade ≤ 10): 8 → ['card_brand', 'card_type', 'customer_risk_rating', 'device_ip_country_code', 'event_type', 'pos_environment', 'tipo_transacao', 'modalidade_transacao']
  Descartadas (cardinalidade > 10): 27


Shape após OHE: (132274, 186)


Das 35 colunas `object` remanescentes após o tratamento de nulidade, **8 tinham cardinalidade ≤ 10** e foram submetidas a OHE: `card_brand`, `card_type`, `customer_risk_rating`, `device_ip_country_code`, `event_type`, `pos_environment`, `tipo_transacao` e `modalidade_transacao`. Essas são exatamente as colunas identificadas na Etapa 3 como discriminantes. As outras **27 colunas `object`** com cardinalidade acima de 10 foram descartadas. O OHE com `drop_first=True` gerou colunas binárias sem dummy trap. O dataset reduz de 210 para **186 colunas**.

### 4.5 Consolidação e verificação final

O bloco monta o dataset final `df_model`, normaliza o alvo para `int8`, confirma ausência total de nulos, registra o shape final e a distribuição do alvo, e lista todas as features disponíveis para modelagem.

In [20]:
# Normaliza alvo para int8
df_prep['feedback_label'] = df_prep['feedback_label'].astype('int8')

# Converte qualquer Int8 (nullable) para int8 padrão (sem nulos remanescentes)
for col in df_prep.select_dtypes(include='Int8').columns:
    df_prep[col] = df_prep[col].astype('int8')

df_model = df_prep.copy()

# Verificação final
n_nulos = df_model.isnull().sum().sum()
n_pos   = df_model['feedback_label'].sum()
n_total = len(df_model)
features = [c for c in df_model.columns if c != 'feedback_label']

print(f'=== df_model ===')
print(f'Shape       : {df_model.shape}')
print(f'Nulos totais: {n_nulos}')
print(f'Alvo positivo (fraude=1): {n_pos} ({n_pos/n_total*100:.4f}%)')
print(f'Alvo negativo (fraude=0): {n_total - n_pos} ({(n_total-n_pos)/n_total*100:.4f}%)')
print(f'Número de features: {len(features)}')
print()
print('--- Dtypes finais ---')
print(df_model.dtypes.value_counts().rename_axis('dtype').reset_index(name='n').to_string(index=False))
print()
print('--- Lista de features ---')
for i, f in enumerate(sorted(features), 1):
    print(f'  {i:3d}. {f}')

=== df_model ===
Shape       : (132274, 186)
Nulos totais: 0
Alvo positivo (fraude=1): 116 (0.0877%)
Alvo negativo (fraude=0): 132158 (99.9123%)
Número de features: 185

--- Dtypes finais ---
  dtype   n
   int8 104
float64  82

--- Lista de features ---
    1. acceptor_latitude
    2. acceptor_latitude_missing
    3. acceptor_longitude
    4. acceptor_longitude_missing
    5. acceptor_mastercard_risk
    6. acceptor_mastercard_risk_missing
    7. acceptor_risk
    8. acceptor_risky_flag
    9. acceptor_risky_flag_missing
   10. acceptor_visa_risk
   11. acceptor_visa_risk_missing
   12. account_associated_cards
   13. account_associated_cards_missing
   14. account_available_credit
   15. account_balance
   16. account_bank_branch_id
   17. account_bank_branch_id_missing
   18. account_bank_name
   19. account_bank_name_missing
   20. account_bic
   21. account_bic_missing
   22. account_branch
   23. account_branch_missing
   24. account_cash_advance_balance
   25. account_cash_advan

`df_model` tem **132.274 registros, 186 colunas, zero nulos**. O alvo mantém a distribuição original: 116 fraudes (0,0877%) e 132.158 não-fraudes. O dataset possui **185 features** — 82 em `float64` (numéricas contínuas) e 103 em `int8` (binárias: booleanos convertidos, indicadores de ausência e dummies de OHE). A lista de features inclui os principais sinais identificados na Etapa 3: `cp_card_term_vel_1h`, `cp_card_prev_term_dist_1h`, `cardid_count_1m`, `fraud_score`, `visa_fraud_score`, `mc_fraud_score`, `amount`, `hora_transacao`, `modalidade_transacao_cartao_presente`, `pos_environment_POS`, `card_type_debito`, `device_ip_country_code_ZZ`, entre outros.

---

## Síntese da Etapa 4

### Principais achados

`df_model` está pronto para modelagem: 132.274 registros, 185 features, zero nulos, alvo com 116 positivos (0,0877%). As 12 colunas inutilizáveis foram removidas (8 constantes, 1 identificadora, 3 leakage pós-evento). As 22 colunas datetime foram descartadas após extração de `hora_transacao`, `mes_transacao` e `dow_transacao`. As 17 colunas booleanas foram convertidas para `int8`. Colunas com nulidade acima de 30% receberam indicadores de ausência binários (75 adicionados); as 152 colunas categóricas esparsas (> 30% nulos) foram descartadas. As 8 colunas categóricas densas com cardinalidade ≤ 10 foram codificadas por OHE; as 27 com cardinalidade superior foram descartadas.

### Pontos de atenção

O dataset está restrito ao subconjunto de 132.274 registros disponível. A concentração temporal das fraudes em jan–mar/2025 (identificada na Etapa 3) persiste e condiciona a estratégia de avaliação na Etapa 5: validação cruzada estratificada será preferível a particionamento temporal puro, dado que os meses restantes não possuem labels confirmados. Os 75 indicadores de ausência aumentam a dimensionalidade e devem ser avaliados quanto à contribuição real no modelo — features sem variância (todos ausentes ou todos presentes) devem ser removidas antes do treino. A feature `hora_transacao` captura o padrão de madrugada identificado na Etapa 3 e deve ter importância relevante no modelo.

## Etapa 5 — Complementos finais da análise exploratória

### Objetivo

Esta etapa fecha os pontos em aberto identificados nas sínteses das Etapas 3 e 4, e prepara o artefato final da análise exploratória. Quatro tarefas independentes são realizadas: investigação dos registros sem timestamp válido, remoção de features de variância zero em `df_model`, análise de correlação de Spearman entre features e o alvo, e exportação do dataset preparado em formato Parquet.

### O que será feito

1. **Investigação dos 258 registros sem timestamp:** caracterizar esses registros quanto a período de ocorrência, taxa de fraude e padrão de nulidade, e decidir se devem ser mantidos ou removidos.
2. **Remoção de features de variância zero em `df_model`:** identificar e remover colunas constantes que possam ter sobrevivido ao pipeline da Etapa 4, atualizando a lista de features.
3. **Correlação de Spearman entre features e o alvo:** calcular a correlação de Spearman de todas as features numéricas de `df_model` com `feedback_label`, ordenar por valor absoluto e exibir as top 20 — complementando a análise de separação de distribuição da Etapa 3.
4. **Exportação em Parquet:** salvar `df_model` em `../data/df_model.parquet`, preservando os tipos exatos, para uso nas etapas de modelagem sem necessidade de reexecutar o pipeline.

### Por que isso é necessário neste ponto

Features de variância zero são ruído puro — aumentam o custo computacional sem contribuição preditiva e podem causar instabilidade numérica em alguns algoritmos. A correlação de Spearman fornece uma ordenação mais formal de relevância do que a razão de medianas, sendo monotônica e robusta a outliers. O Parquet consolida o resultado do pipeline e torna as etapas de modelagem independentes da reexecução da EDA.

### 5.1 Investigação dos 258 registros sem timestamp

O bloco identifica, a partir de `df_work`, os registros cujo `event_occurred_at` não pôde ser convertido para `datetime64` (produzindo `NaT`). Para cada um, verifica: qual é o valor original na coluna (em `df`), qual a taxa de fraude no grupo, e qual o grau médio de nulidade. Essa informação determina se os registros devem ser mantidos em `df_model` ou removidos.

In [21]:
# Registros com NaT em event_occurred_at (df_work já tem a coluna convertida)
mask_nat = df_work['event_occurred_at'].isnull()
n_nat = mask_nat.sum()
print(f'Registros com event_occurred_at = NaT: {n_nat}')

# Valor original em df
nat_idx = df_work.index[mask_nat]
orig_vals = df.loc[nat_idx, 'event_occurred_at'].value_counts(dropna=False)
print('\nValores originais em df para esses registros:')
print(orig_vals.to_string())

# Taxa de fraude no grupo NaT vs. restante
taxa_nat   = df_work.loc[mask_nat,  'feedback_label'].astype(bool).mean() * 100
taxa_resto = df_work.loc[~mask_nat, 'feedback_label'].astype(bool).mean() * 100
print(f'\nTaxa de fraude — NaT: {taxa_nat:.4f}%  |  restante: {taxa_resto:.4f}%')

# Nulidade média por registro no grupo NaT
null_nat_mean   = df_work.loc[mask_nat].isnull().mean(axis=1).mean() * 100
null_resto_mean = df_work.loc[~mask_nat].isnull().mean(axis=1).mean() * 100
print(f'Nulidade média por linha — NaT: {null_nat_mean:.1f}%  |  restante: {null_resto_mean:.1f}%')

Registros com event_occurred_at = NaT: 258

Valores originais em df para esses registros:
event_occurred_at
NaN    258

Taxa de fraude — NaT: 0.0000%  |  restante: 0.0879%


Nulidade média por linha — NaT: 60.5%  |  restante: 60.2%


Os 258 registros sem timestamp têm como valor original em `df` apenas `NaN` — o campo estava ausente desde a origem, não é um problema de formato de string que falhou no parse. A taxa de fraude no grupo é **0,0000%** (nenhuma fraude), comparado a 0,0879% no restante. A nulidade média por linha é 60,5%, praticamente idêntica à do restante (60,2%). Os registros não apresentam nenhuma característica distinta que os torne informativos ou problemáticos — são simplesmente registros sem timestamp. Como não carregam nenhuma fraude e sua nulidade é homogênea, **não há justificativa para removê-los de `df_model`**: a imputação pela mediana aplicada na Etapa 4 para `hora_transacao`, `mes_transacao` e `dow_transacao` é aceitável para esse grupo residual.

### 5.2 Remoção de features de variância zero em `df_model`

O bloco identifica em `df_model` todas as colunas (exceto o alvo) com variância zero — colunas cujo valor é constante em todos os 132.274 registros. Isso pode ocorrer, por exemplo, em indicadores de ausência para os quais nenhum registro teve o campo preenchido, ou colunas booleanas que assumiram apenas um valor após a imputação. Essas colunas são removidas e `df_model` é atualizado.

In [22]:
features_all = [c for c in df_model.columns if c != 'feedback_label']

zero_var = [c for c in features_all if df_model[c].nunique() <= 1]
print(f'Features com variância zero: {len(zero_var)}')
if zero_var:
    print(zero_var)
    df_model = df_model.drop(columns=zero_var)
    print(f'Shape após remoção: {df_model.shape}')
else:
    print('Nenhuma coluna constante encontrada — df_model inalterado.')
    print(f'Shape: {df_model.shape}')

features_final = [c for c in df_model.columns if c != 'feedback_label']
print(f'Features finais: {len(features_final)}')

Features com variância zero: 0
Nenhuma coluna constante encontrada — df_model inalterado.
Shape: (132274, 186)
Features finais: 185


Nenhuma feature de variância zero foi encontrada em `df_model`. Os 75 indicadores de ausência adicionados na Etapa 4 têm todos variância positiva, o que significa que para cada um deles há ao menos um registro com o campo ausente e ao menos um com o campo preenchido — exatamente o comportamento esperado para indicadores com sentido informativo. `df_model` permanece com 186 colunas e 185 features.

### 5.3 Correlação de Spearman entre features e o alvo

A correlação de Spearman mede a associação monotônica entre cada feature e `feedback_label` sem pressupor linearidade nem distribuição normal — propriedades importantes dado o forte desbalanceamento e as distribuições assimétricas observadas na Etapa 3. O bloco calcula a correlação de Spearman de todas as features numéricas de `df_model` com o alvo e exibe as top 20 em valor absoluto, que são as candidatas mais fortes a features discriminantes.

In [23]:
from scipy.stats import spearmanr

y = df_model['feedback_label'].astype(float)

corr_rows = []
for col in features_final:
    x = df_model[col].astype(float)
    # Usa apenas pares sem NaN (não deve haver, mas por segurança)
    mask = x.notna() & y.notna()
    if mask.sum() < 10:
        continue
    r, p = spearmanr(x[mask], y[mask])
    corr_rows.append({'feature': col, 'spearman_r': round(r, 5), 'p_value': p})

corr_df = (
    pd.DataFrame(corr_rows)
    .assign(abs_r=lambda d: d['spearman_r'].abs())
    .sort_values('abs_r', ascending=False)
    .reset_index(drop=True)
)

print('--- Top 20 features por |correlação de Spearman| com feedback_label ---')
print(corr_df[['feature', 'spearman_r', 'p_value']].head(20).to_string(index=False))

--- Top 20 features por |correlação de Spearman| com feedback_label ---
                             feature  spearman_r       p_value
           device_ip_country_code_ZZ     0.50838  0.000000e+00
                     cardid_count_1m     0.07424 5.504617e-161
                 cp_card_term_vel_1h     0.05163  9.011277e-79
                         fraud_score     0.05039  4.101535e-75
         digitalwallettoken_count_3m     0.04730  2.197686e-66
                    visa_fraud_score     0.04686  3.278828e-65
                      mc_fraud_score     0.04685  3.499209e-65
           cp_card_prev_term_dist_1h     0.04425  2.436844e-58
                      hora_transacao    -0.04176  3.788497e-52
                              amount     0.04041  6.301874e-49
                       acceptor_risk     0.04021  1.800987e-48
                       mes_transacao    -0.03846  1.768637e-44
                 pos_environment_POS    -0.03477  1.123342e-36
modalidade_transacao_cartao_presente    -0.034

A correlação de Spearman confirma e ordena formalmente os sinais identificados na Etapa 3. `device_ip_country_code_ZZ` lidera com r = 0,508 — correlação notavelmente alta para um dataset tão desbalanceado, explicada pelo fato de que 100% dos 30 registros com IP de origem `ZZ` são fraudes. As features de velocidade e contagem (`cardid_count_1m`, `cp_card_term_vel_1h`) e os scores de fraude (`fraud_score`, `visa_fraud_score`, `mc_fraud_score`) formam o segundo grupo relevante, com r entre 0,047 e 0,074. `hora_transacao` aparece com r = −0,042 (negativo: horas menores, i.e. madrugada, associadas a fraude) e `mes_transacao` com r = −0,038 (meses menores, i.e. jan–mar, onde estão todas as fraudes). A partir da 19ª posição (`terminal_poc_flag_missing` e `account_status_missing`), os p-valores já superam 0,01 — essas features têm correlação residual e estatisticamente fraca. As 185 features restantes não atingiram o limiar de inclusão nas top 20, confirmando que o poder discriminante está concentrado num subconjunto pequeno.

### 5.4 Exportação em Parquet

O bloco salva `df_model` no formato Parquet, que preserva os dtypes exatos (`float64`, `int8`), permite leitura colunar eficiente e é ordens de magnitude mais rápido de carregar do que o CSV original de 232 MB. O arquivo será salvo em `../data/df_model.parquet`. Após a gravação, verifica-se o tamanho do arquivo e recarrega-se o dataset para confirmar integridade de shape e tipos.

In [24]:
import os

PARQUET_PATH = '../data/df_model.parquet'
df_model.to_parquet(PARQUET_PATH, index=False)

size_mb = os.path.getsize(PARQUET_PATH) / 1024**2
print(f'Arquivo salvo: {PARQUET_PATH}')
print(f'Tamanho: {size_mb:.1f} MB')

# Verificação de integridade
df_check = pd.read_parquet(PARQUET_PATH)
assert df_check.shape == df_model.shape, 'Shape diverge!'
assert df_check.isnull().sum().sum() == 0, 'Nulos encontrados!'
print(f'Verificação: shape {df_check.shape} — OK')
print(f'Dtypes carregados:')
print(df_check.dtypes.value_counts().rename_axis('dtype').reset_index(name='n').to_string(index=False))

Arquivo salvo: ../data/df_model.parquet
Tamanho: 16.5 MB


Verificação: shape (132274, 186) — OK
Dtypes carregados:
  dtype   n
   int8 104
float64  82


`df_model.parquet` foi salvo com **16,5 MB** — redução de mais de 93% em relação ao CSV original de 232 MB. A verificação de integridade confirmou shape idêntico (132.274 × 186), zero nulos e dtypes preservados (`int8` e `float64`). O arquivo está disponível em `../data/df_model.parquet` e pode ser carregado diretamente na etapa de modelagem com `pd.read_parquet()`.

---

## Síntese da Etapa 5 — Encerramento da análise exploratória

### Principais achados

Os 258 registros sem timestamp têm `NaN` original no campo — não é falha de parsing — e nenhuma fraude associada; foram mantidos em `df_model` com imputação pela mediana para as features temporais. Nenhuma feature de variância zero foi encontrada: todos os 75 indicadores de ausência são informativos. A correlação de Spearman identificou um ranking formal de relevância: `device_ip_country_code_ZZ` (r = 0,508) é a feature mais discriminante, seguida de `cardid_count_1m` (r = 0,074), `cp_card_term_vel_1h` (r = 0,052) e os três scores de fraude (~r = 0,047–0,050). A partir da 19ª posição, as correlações são estatisticamente fracas (p > 0,01). O dataset final `df_model.parquet` (16,5 MB, 132.274 × 186, zero nulos) está exportado e pronto para modelagem.

### Pontos de atenção

O forte desbalanceamento do alvo (0,088% positivos) e a concentração das fraudes em jan–mar/2025 são as duas restrições que mais condicionam a modelagem. A métrica de avaliação deve ser PR-AUC ou F1 ponderado — accuracy é inadequada neste contexto. A estratégia de validação recomendada é validação cruzada estratificada sobre os registros de jan–mar/2025, que são os únicos com labels confirmados. As features com maior poder discriminante (velocidade, distância geográfica, scores de fraude e variáveis de ambiente como ECOMMERCE e cartão não presente) foram todas mapeadas e estão disponíveis em `df_model`.